# Pipeline completo de series de tiempo con PyTorch Lightning y Keras/JAX



La primera parte utiliza:

- PyTorch
- `Dataset`
- `DataLoader`
- transformaciones
- PyTorch Lightning
- TensorBoard
- EarlyStopping
- ModelCheckpoint

La segunda parte reutiliza el mismo pipeline de datos y los mismos `DataLoader`, pero construye y entrena una red con:

- Keras 3
- backend **JAX**

## Problema

A partir de los últimos 12 meses:

$
[x_{t-11}, ..., x_t]
$

predeciremos:

$
x_{t+1}
$


> La división temporal es cronológica: nunca se mezclan observaciones futuras dentro de train.

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "jax"

import keras
from keras import layers
print("Keras:", keras.__version__)
print("Backend:", keras.backend.backend())

In [ ]:
# Si fuera necesario:
# !pip install torch lightning tensorboard
# !pip install "keras>=3" jax jaxlib

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import lightning.pytorch as pl
from lightning.pytorch.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    LearningRateMonitor
)
from lightning.pytorch.loggers import TensorBoardLogger

print("PyTorch:", torch.__version__)
print("Lightning:", pl.__version__)

SEED = 42
pl.seed_everything(SEED, workers=True)

# 1. Dataset Sunspots

# 2. División temporal

Usamos:

- 70% train
- 15% validation
- 15% test

La división **no se aleatoriza**.

Esto es fundamental en forecasting.

In [ ]:
CSV_PATH=r"C:\Users\javier\Documents\deep_learning_2026_phd\deep-learning-i\clase1\sunspots.csv"

df = pd.read_csv(CSV_PATH)
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)
values = df[
    "Monthly Mean Total Sunspot Number"
].values

dates = df["Date"].values
N = len(df)
train_end = int(0.70 * N)
val_end = int(0.85 * N)
plt.figure(figsize=(14, 5))
plt.plot(
    dates[:train_end],
    values[:train_end],
    label="Train")
plt.plot(
    dates[train_end:val_end],
    values[train_end:val_end],
    label="Validation")
plt.plot(
    dates[val_end:],
    values[val_end:],
    label="Test")

# Líneas que marcan los cortes
plt.axvline(
    dates[train_end],
    linestyle="--",
    alpha=0.7
)

plt.axvline(
    dates[val_end],
    linestyle="--",
    alpha=0.7
)

plt.xlabel("Date")
plt.ylabel("Monthly Mean Total Sunspot Number")
plt.title( "Sunspots Time Series - Train / Validation / Test")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
class SunspotsDataset(Dataset):

    def __init__(
        self,
        csv_path,
        split="train",
        window_size=24,
        train_ratio=0.70,
        val_ratio=0.15,
        transform=None,
        mean=None,
        std=None
    ):

        self.window_size = window_size
        self.transform = transform

        df = pd.read_csv(csv_path)
        df["Date"] = pd.to_datetime(
            df["Date"]
        )
        df = df.sort_values(
            "Date"
        ).reset_index(drop=True)

        values = df[
            "Monthly Mean Total Sunspot Number"
        ].values.astype(np.float32)

        dates = df[
            "Date"
        ].values

 
        N = len(values)

        train_end = int(
            train_ratio * N
        )

        val_end = int(
            (train_ratio + val_ratio) * N
        )

        train_values = values[:train_end]

        if mean is None:
            mean = train_values.mean()

        if std is None:
            std = train_values.std()

        self.mean = float(mean)
        self.std = float(std)

        if split == "train":

            split_values = values[
                :train_end
            ]

            split_dates = dates[
                :train_end
            ]

        elif split == "val":

            split_values = values[
                train_end - window_size:
                val_end
            ]

            split_dates = dates[
                train_end - window_size:
                val_end
            ]

        elif split == "test":

   
            split_values = values[
                val_end - window_size:
            ]

            split_dates = dates[
                val_end - window_size:
            ]

        else:

            raise ValueError(
                "split debe ser 'train', 'val' o 'test'"
            )


        split_values = (
            split_values - self.mean
        ) / self.std

        self.values = torch.tensor(
            split_values,
            dtype=torch.float32
        )

        self.dates = split_dates
    def __len__(self):

        return (
            len(self.values)
            - self.window_size
        )


    def __getitem__(self, idx):

        # Ventana
        x = self.values[
            idx :
            idx + self.window_size
        ].clone()

        # siguiente punto temporal
        y = self.values[
            idx + self.window_size
        ].clone()

        if self.transform is not None:

            x = self.transform(x)

        return x, y



# 3. Normalización: estadísticas calculadas solo con train

Calculamos:

$
z = \frac{x-\mu_{train}}{\sigma_{train}}
$

La validación y el test reutilizan exactamente la media y desviación estándar de train.

Esto evita utilizar información del futuro durante el preprocesamiento.

# 4. Transformaciones en PyTorch

Para mostrar un pipeline reusable definimos transformaciones explícitas.

El train utilizará una perturbación gaussiana muy pequeña sobre la ventana de entrada.

Validation y test no reciben transformaciones aleatorias.

In [ ]:
class Compose:
    def __init__(self, transforms):
        self.transforms = transforms

    def __call__(self, x):
        for transform in self.transforms:
            x = transform(x)

        return x


class AddGaussianNoise:
    def __init__(self, std=0.01):
        self.std = std

    def __call__(self, x):
        noise = torch.randn_like(x) * self.std

        return x + noise
train_transform = Compose([
    AddGaussianNoise(std=0.01)
])
eval_transform = None

# 5. `Dataset` personalizado para crear ventanas

El `Dataset` recibe una serie estandarizada.

Para cada índice `i` devuelve:

```python
x = serie[i : i + WINDOW_SIZE]
y = serie[i + WINDOW_SIZE]
```

Así, `DataLoader` puede encargarse posteriormente de:

- batching;
- shuffle;
- paralelización.

In [ ]:
WINDOW_SIZE = 24

train_dataset = SunspotsDataset(
    csv_path=CSV_PATH,
    split="train",
    window_size=WINDOW_SIZE,
    transform=train_transform
)

val_dataset = SunspotsDataset(
    csv_path=CSV_PATH,
    split="val",
    window_size=WINDOW_SIZE,
    transform=None,

    mean=train_dataset.mean,
    std=train_dataset.std
)

test_dataset = SunspotsDataset(
    csv_path=CSV_PATH,
    split="test",
    window_size=WINDOW_SIZE,
    transform=None,

    mean=train_dataset.mean,
    std=train_dataset.std
)


def inverse_standardize(x):
    return (
        x * train_dataset.std
        + train_dataset.mean
    )

In [ ]:
BATCH_SIZE = 32


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


In [ ]:
for x_batch, y_batch in train_loader:
    print("X:", x_batch.shape, "y:", y_batch.shape)
    break

## Ventanas de validation y test con contexto

Para predecir el primer punto de validation necesitamos los últimos `WINDOW_SIZE` puntos de train.

Igualmente, para test necesitamos los últimos puntos anteriores al comienzo de test.

Esto permite evaluar todos los puntos sin introducir observaciones futuras en las entradas.

# 7. Red neuronal densa

No usaremos RNN, LSTM ni convoluciones.

Cada ventana de 12 meses se interpreta como un vector de 12 características.

Arquitectura:

```text
12
 ↓
Dense(64)
 ↓
BatchNorm
 ↓
ReLU
 ↓
Dropout
 ↓
Dense(32)
 ↓
BatchNorm
 ↓
ReLU
 ↓
Dense(1)
```

In [ ]:
class DenseForecaster(nn.Module):
    def __init__(self, window_size=12):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(window_size, 64, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.15),

            nn.Linear(64, 32, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(1)

base_model = DenseForecaster(WINDOW_SIZE)
print(base_model)

# 8. `LightningModule`: equivalente organizado de `compile + fit`

Lightning encapsula:

- modelo;
- loss;
- métricas;
- optimizador;
- scheduler;
- training step;
- validation step;
- test step;
- predict step.

El `Trainer` se encargará del loop.

In [ ]:
class LitTimeSeriesForecaster(pl.LightningModule):
    def __init__(
        self,
        window_size=12,
        learning_rate=1e-3,
        weight_decay=1e-4
    ):
        super().__init__()

        self.save_hyperparameters()

        self.model = DenseForecaster(
            window_size=window_size
        )

        self.loss_fn = nn.MSELoss()

    def forward(self, x):
        return self.model(x)

    def _shared_step(self, batch):
        x, y = batch

        pred = self(x)

        loss = self.loss_fn(pred, y)
        mae = torch.mean(torch.abs(pred - y))

        return loss, mae, pred, y

    def training_step(self, batch, batch_idx):
        loss, mae, _, _ = self._shared_step(batch)

        self.log(
            "train_loss",
            loss,
            on_step=False,
            on_epoch=True,
            prog_bar=True
        )

        self.log(
            "train_mae",
            mae,
            on_step=False,
            on_epoch=True
        )

        return loss

    def validation_step(self, batch, batch_idx):
        loss, mae, _, _ = self._shared_step(batch)

        self.log(
            "val_loss",
            loss,
            on_step=False,
            on_epoch=True,
            prog_bar=True
        )

        self.log(
            "val_mae",
            mae,
            on_step=False,
            on_epoch=True,
            prog_bar=True
        )

    def test_step(self, batch, batch_idx):
        loss, mae, _, _ = self._shared_step(batch)

        self.log(
            "test_loss",
            loss,
            on_epoch=True
        )

        self.log(
            "test_mae",
            mae,
            on_epoch=True,
            prog_bar=True
        )

    def predict_step(
        self,
        batch,
        batch_idx,
        dataloader_idx=0
    ):
        x, y = batch
        pred = self(x)

        return {
            "prediction": pred,
            "target": y
        }

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.learning_rate,
            weight_decay=self.hparams.weight_decay
        )

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=3,
            min_lr=1e-6
        )

        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss"
            }
        }

# 9. TensorBoard, EarlyStopping y Checkpoint

Lightning usa callbacks de la misma forma conceptual que Keras.

In [ ]:
TORCH_ARTIFACT_DIR = Path("timeseries_lightning")
TORCH_ARTIFACT_DIR.mkdir(exist_ok=True)

logger = TensorBoardLogger(
    save_dir=str(TORCH_ARTIFACT_DIR),
    name="tensorboard_logs"
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=8,
    min_delta=1e-4,
    verbose=True
)

checkpoint = ModelCheckpoint(
    dirpath=str(TORCH_ARTIFACT_DIR / "checkpoints"),
    filename="best-{epoch:02d}-{val_loss:.4f}",
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    save_last=True
)

lr_monitor = LearningRateMonitor(
    logging_interval="epoch"
)

# 10. `Trainer.fit()`

`Trainer.fit()` se encarga automáticamente de:

- forward;
- backward;
- actualización del optimizador;
- cambio `train()/eval()`;
- validación;
- callbacks;
- logging;
- CPU/GPU.

In [ ]:
lit_model = LitTimeSeriesForecaster(
    window_size=WINDOW_SIZE,
    learning_rate=1e-3,
    weight_decay=1e-4
)

trainer = pl.Trainer(
    max_epochs=100,
    accelerator="auto",
    devices=1,
    logger=logger,
    callbacks=[
        early_stopping,
        checkpoint,
        lr_monitor
    ],
    deterministic=True,
    log_every_n_steps=1
)

trainer.fit(
    lit_model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader
)

# 11. TensorBoard

En Jupyter/Colab:

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir timeseries_lightning

# 12. Mejor checkpoint

In [ ]:
print("Mejor checkpoint:")
print(checkpoint.best_model_path)

print("\nMejor val_loss:")
print(checkpoint.best_model_score)

# 13. Cargar el mejor modelo

In [ ]:
best_lit_model = LitTimeSeriesForecaster.load_from_checkpoint(
    checkpoint.best_model_path
)

best_lit_model.eval()

# 14. Evaluación con `Trainer.test()`

In [ ]:
test_results = trainer.test(
    best_lit_model,
    dataloaders=test_loader
)

test_results

# 15. Predicción con `Trainer.predict()`

In [ ]:
prediction_batches = trainer.predict(
    best_lit_model,
    dataloaders=test_loader
)

pred_scaled = torch.cat([
    batch["prediction"].detach().cpu()
    for batch in prediction_batches
]).numpy()

true_scaled = torch.cat([
    batch["target"].detach().cpu()
    for batch in prediction_batches
]).numpy()

pred_suspots = inverse_standardize(pred_scaled)
true_suspots = inverse_standardize(true_scaled)

print(pred_suspots[:5])
print(true_suspots[:5])

# 16. Métricas en unidades originales

In [ ]:
mae = np.mean(
    np.abs(pred_suspots - true_suspots)
)

rmse = np.sqrt(
    np.mean(
        (pred_suspots - true_suspots) ** 2
    )
)

print(f"MAE  = {mae:.3f} suspots")
print(f"RMSE = {rmse:.3f} suspots")

# 17. Visualización del forecasting

In [ ]:
test_dates = dates[val_end:]

plt.figure(figsize=(13, 5))

plt.plot(
    test_dates,
    true_suspots,
    marker="o",
    label="Real"
)

plt.plot(
    test_dates,
    pred_suspots,
    marker="o",
    label="Predicción"
)

plt.xlabel("Fecha")
plt.ylabel("Pasajeros")
plt.title("PyTorch Lightning - Test")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# PARTE II — El mismo problema usando Keras con backend JAX

Keras 3 permite utilizar:

```python
KERAS_BACKEND="jax"
```

y seguir usando:

- `keras.Sequential`
- `keras.layers`
- `compile()`
- `fit()`
- `evaluate()`
- `predict()`
- callbacks

Además, Keras 3 puede recibir directamente un **PyTorch `DataLoader`**, independientemente del backend seleccionado.

Por tanto reutilizaremos:

```python
train_loader
val_loader
test_loader
```

creados anteriormente.

## Importante: seleccionar backend antes de importar Keras

Esta celda debe ejecutarse **antes del primer `import keras` del kernel**.

Si ya importaste Keras con otro backend en el mismo kernel, reinicia el kernel y ejecuta el notebook en orden.

In [ ]:
print("Keras:", keras.__version__)
print("Backend:", keras.backend.backend())

# 18. Modelo denso Keras/JAX

Usamos una arquitectura conceptualmente equivalente a la utilizada con PyTorch.

In [ ]:
jax_model = keras.Sequential([
    keras.Input(shape=(WINDOW_SIZE,)),

    layers.Dense(
        64,
        use_bias=False
    ),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.15),

    layers.Dense(
        32,
        use_bias=False
    ),
    layers.BatchNormalization(),
    layers.Activation("relu"),

    layers.Dense(1)
])

jax_model.summary()

# 19. `compile()` con backend JAX

La API es la misma que en Keras con TensorFlow, pero las operaciones del modelo se ejecutan mediante JAX.

In [ ]:
jax_model.compile(
    optimizer=keras.optimizers.AdamW(
        learning_rate=1e-3,
        weight_decay=1e-4
    ),

    loss=keras.losses.MeanSquaredError(),

    metrics=[
        keras.metrics.MeanAbsoluteError(
            name="mae"
        )
    ]
)

# 20. Callbacks Keras/JAX

In [ ]:
JAX_ARTIFACT_DIR = Path("timeseries_keras_jax")
JAX_ARTIFACT_DIR.mkdir(exist_ok=True)

jax_best_path = (
    JAX_ARTIFACT_DIR /
    "best_timeseries_jax.keras"
)

jax_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=8,
        min_delta=1e-4,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ModelCheckpoint(
        filepath=str(jax_best_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),

    keras.callbacks.TensorBoard(
        log_dir=str(
            JAX_ARTIFACT_DIR /
            "tensorboard_logs"
        )
    )
]

# 21. `fit()` usando directamente los `DataLoader` de PyTorch

No convertimos los datos a NumPy ni creamos otro pipeline.

Keras recibe directamente:

```python
train_loader
val_loader
```

In [ ]:
jax_history = jax_model.fit(
    train_loader,
    validation_data=val_loader,
    epochs=100,
    callbacks=jax_callbacks,
    verbose=1
)

# 22. History

In [ ]:
plt.figure(figsize=(9, 4))

plt.plot(
    jax_history.history["loss"],
    label="train"
)

plt.plot(
    jax_history.history["val_loss"],
    label="validation"
)

plt.xlabel("Época")
plt.ylabel("MSE")
plt.title("Keras + JAX")
plt.legend()
plt.show()

# 23. Cargar el mejor modelo Keras/JAX

In [ ]:
best_jax_model = keras.models.load_model(
    jax_best_path
)

best_jax_model.summary()

# 24. Evaluación utilizando `test_loader`

In [ ]:
jax_test_results = best_jax_model.evaluate(
    test_loader,
    return_dict=True,
    verbose=1
)

jax_test_results

# 25. `DataLoader` para `predict()`

Para `evaluate()` necesitamos `(x, y)`.

Para `predict()` crearemos un wrapper que devuelve únicamente `x`.

In [ ]:
class InputsOnlyDataset(Dataset):
    def __init__(self, base_dataset):
        self.base_dataset = base_dataset

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        x, _ = self.base_dataset[idx]

        return (x,)



jax_predict_loader = DataLoader(
    InputsOnlyDataset(test_dataset),
    batch_size=BATCH_SIZE,
    shuffle=False
)


In [ ]:
jax_pred_scaled = best_jax_model.predict(
    jax_predict_loader,
    verbose=0
).squeeze()



jax_pred_suspots = inverse_standardize(
    jax_pred_scaled
)


jax_mae = np.mean(
    np.abs(
        jax_pred_suspots -
        true_suspots
    )
)

jax_rmse = np.sqrt(
    np.mean(
        (
            jax_pred_suspots -
            true_suspots
        ) ** 2
    )
)

print(f"JAX MAE  = {jax_mae:.3f}")
print(f"JAX RMSE = {jax_rmse:.3f}")

# 26. Comparación PyTorch Lightning vs Keras/JAX

In [ ]:
plt.figure(figsize=(13, 5))
test_dates = dates[val_end:]
plt.plot(
    test_dates,
    true_suspots,
    marker="o",
    label="Real"
)

plt.plot(
    test_dates,
    pred_suspots,
    marker="o",
    label="PyTorch Lightning"
)

plt.plot(
    test_dates,
    jax_pred_suspots,
    marker="o",
    label="Keras + JAX"
)

plt.xlabel("Fecha")
plt.ylabel("Pasajeros")
plt.title("Comparación de predicciones")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 27. Comprobar la interoperabilidad del `DataLoader`

El pipeline de datos sigue siendo PyTorch:

In [ ]:
x_batch, y_batch = next(iter(train_loader))

print("Input batch:")
print(type(x_batch), x_batch.shape)

print("\nTarget batch:")
print(type(y_batch), y_batch.shape)

jax_output = jax_model.predict(
    x_batch,
    verbose=0
)

print("\nSalida del modelo Keras/JAX:")
print(type(jax_output), jax_output.shape)

# Resumen del pipeline

## PyTorch Lightning

```text
AirPassengers
      ↓
split cronológico
      ↓
estadísticas de train
      ↓
standardization
      ↓
TimeSeriesWindowDataset
      ↓
transformaciones
      ↓
DataLoader
      ↓
Dense + BatchNorm
      ↓
LightningModule
      ↓
configure_optimizers
      ↓
Trainer.fit
      ↓
TensorBoard
EarlyStopping
ModelCheckpoint
LR scheduler
      ↓
load_from_checkpoint
      ↓
Trainer.test
      ↓
Trainer.predict
      ↓
inverse transform
      ↓
MAE / RMSE / gráfica
```

## Keras + JAX

```text
MISMOS PyTorch DataLoader
      ↓
Keras Sequential
      ↓
backend JAX
      ↓
compile
      ↓
fit
      ↓
callbacks
      ↓
load_model
      ↓
evaluate
      ↓
predict
      ↓
inverse transform
```

## Buenas prácticas aplicadas

- división cronológica;
- no se usa test durante entrenamiento;
- normalización ajustada únicamente con train;
- ventanas creadas en un `Dataset`;
- `DataLoader` para batching;
- transformación separada entre train y evaluación;
- Batch Normalization;
- callbacks;
- TensorBoard;
- EarlyStopping;
- checkpoints;
- carga del mejor modelo;
- evaluación en escala normalizada y en unidades originales;
- inferencia separada;
- interoperabilidad entre PyTorch `DataLoader` y Keras/JAX.